In [ ]:
## knockdown code

In [ ]:
#aggreagation

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from tqdm import tqdm

# ==========================================
# 1. SETUP & PATHS (MULTI-PLATE)
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
          "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
          "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

# Paths for the two separate outputs
OUTPUT_CSV_MEDIAN = os.path.join(PROJECT_ROOT,"10marchecht", "aggregated_wells_median.csv")
OUTPUT_CSV_STD = os.path.join(PROJECT_ROOT,"10marchecht", "aggregated_wells_std.csv")

CELL_COUNT_THRESHOLD = 0  
TREATMENT_COL = "Treatment"

all_plates_median = []
all_plates_std = []
all_cell_counts = [] 

for plate_id in PLATES:
    print(f"\n--- Processing {plate_id} ---")
    
    FEATURES_BASE = os.path.join(PROJECT_ROOT, "features", plate_id)
    METADATA_PATH = os.path.join(PROJECT_ROOT, "metadata", f"index_{plate_id}.csv")
    
    if not os.path.exists(METADATA_PATH):
        print(f"Skipping {plate_id}: Metadata not found.")
        continue

    meta = pd.read_csv(METADATA_PATH)
    well_storage = {}
    well_to_treatment = {}

    for i in tqdm(meta.index, desc=f"Loading {plate_id}"):
        well_id = f"{plate_id}_{meta.loc[i, 'Metadata_Well']}"
        treatment = str(meta.loc[i, TREATMENT_COL]).strip()
        
        filename = os.path.join(FEATURES_BASE, 
                                str(meta.loc[i, "Metadata_Well"]), 
                                f"{meta.loc[i, 'Metadata_Site']}.npz")
        
        if os.path.isfile(filename):
            try:
                with np.load(filename) as data:
                    cells = data["features"]
                    cells_f = cells[~np.isnan(cells).any(axis=1)]
                    
                    if len(cells_f) > 0:
                        if well_id not in well_storage:
                            well_storage[well_id] = []
                            well_to_treatment[well_id] = treatment
                        well_storage[well_id].append(cells_f)
            except:
                continue

    # --- AGGREGATION & THRESHOLDING STEP ---
    for well_id, feature_list in well_storage.items():
        all_cells_in_well = np.vstack(feature_list)
        well_cell_count = all_cells_in_well.shape[0]
        all_cell_counts.append(well_cell_count)

        if well_cell_count >= CELL_COUNT_THRESHOLD:
            # Calculate both Median and Std Dev
            well_median = np.median(all_cells_in_well, axis=0)
            well_std = np.std(all_cells_in_well, axis=0)
            
            base_info = {
                "Plate": plate_id, 
                "Well_ID": well_id, 
                "Treatment": well_to_treatment[well_id],
                "Cell_Count": well_cell_count
            }
            
            # Create rows for both dataframes
            row_median = base_info.copy()
            row_std = base_info.copy()
            
            for idx in range(len(well_median)):
                row_median[idx] = well_median[idx]
                row_std[idx] = well_std[idx]
                
            all_plates_median.append(row_median)
            all_plates_std.append(row_std)

# Convert to DataFrames
df_median = pd.DataFrame(all_plates_median)
df_std = pd.DataFrame(all_plates_std)

# Helper function to reorder
def reorder_cols(df):
    meta_cols = ["Plate", "Well_ID", "Treatment", "Cell_Count"]
    feat_cols = [c for c in df.columns if c not in meta_cols]
    return df[meta_cols + feat_cols]

df_median = reorder_cols(df_median)
df_std = reorder_cols(df_std)

print(f"\nAggregation complete.")

# ==========================================
# 2. VISUALIZATION: CELL COUNT HISTOGRAM
# ==========================================
counts = np.array(all_cell_counts)
c_mean = np.mean(counts)
c_median = np.median(counts)
c_std = np.std(counts)

plt.figure(figsize=(10, 6))
plt.hist(counts, bins=50, color='skyblue', edgecolor='black', alpha=0.7)

# Create stats text string
stats_text = f'Mean: {c_mean:.2f}\nMedian: {c_median:.2f}\nStd Dev: {c_std:.2f}'
# Place text box in the plot
plt.gca().text(0.95, 0.95, stats_text, transform=plt.gca().transAxes, 
               verticalalignment='top', horizontalalignment='right',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

plt.title('Distribution of Cell Counts per Well (All Plates)')
plt.xlabel('Number of Cells')
plt.ylabel('Frequency (Wells)')
plt.grid(axis='y', alpha=0.3)
plt.show()

# ==========================================
# 3. SAVE TO CSVs
# ==========================================
df_median.to_csv(OUTPUT_CSV_MEDIAN, index=False)
df_std.to_csv(OUTPUT_CSV_STD, index=False)

print(f"Median data saved to: {OUTPUT_CSV_MEDIAN}")
print(f"Std Dev data saved to: {OUTPUT_CSV_STD}")

In [ ]:
#count patches

In [ ]:
import numpy as np
import os
from tqdm import tqdm

PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
          "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
          "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

total_patches = 0
total_files = 0

print("Calculating total patch count...")

for plate in PLATES:
    feature_path = os.path.join(PROJECT_ROOT, "features", plate)
    
    if not os.path.exists(feature_path):
        print(f"Skipping {plate}: Path not found.")
        continue
        
    # Walk through all well subfolders
    for root, dirs, files in os.walk(feature_path):
        for file in files:
            if file.endswith(".npz"):
                file_path = os.path.join(root, file)
                try:
                    with np.load(file_path) as data:
                        # 'features' is the standard key in DeepProfiler npz files
                        # We only need the shape[0] (number of rows/cells)
                        total_patches += data["features"].shape[0]
                        total_files += 1
                except Exception as e:
                    print(f"Could not read {file}: {e}")

print("\n--- Final Statistics ---")
print(f"Total .npz files (sites) processed: {total_files}")
print(f"Total number of patches (cells):     {total_patches:,}")

In [ ]:
#remove mutants less than 5 patches

In [ ]:
import pandas as pd
import os

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
INPUT_CSV = os.path.join(PROJECT_ROOT, "10marchecht", "aggregated_wells_median.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "10marchecht")

# Create the directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. DEFINE YOUR NEW THRESHOLD
STRICT_THRESHOLD = 5 

# 3. LOAD & FILTER
print(f"Loading {INPUT_CSV}...")
df = pd.read_csv(INPUT_CSV)

# Identify the wells that ARE BELOW the threshold
removed_df = df[df['Cell_Count'] < STRICT_THRESHOLD].copy()

# Identify the wells that ARE ABOVE or EQUAL to the threshold
filtered_df = df[df['Cell_Count'] >= STRICT_THRESHOLD].copy()

# 4. REPORT & SAVE
print(f"\n--- Filtering Summary ---")
print(f"Original wells:       {len(df)}")
print(f"Wells kept:           {len(filtered_df)}")
print(f"Wells removed:        {len(removed_df)}")
print(f"-------------------------")

if not removed_df.empty:
    print(f"\n--- LIST OF REMOVED WELLS (Count < {STRICT_THRESHOLD}) ---")
    # We only show the metadata columns for the removed wells
    print(removed_df[['Plate', 'Well_ID', 'Treatment', 'Cell_Count']].to_string(index=False))
else:
    print("\nNo wells were below the threshold.")

# 5. SAVE DATA
OUTPUT_CSV_FILTERED = os.path.join(OUTPUT_DIR, f"aggregated_wells_median_min5.csv")
filtered_df.to_csv(OUTPUT_CSV_FILTERED, index=False)

print(f"\nDone! Filtered data saved to: {OUTPUT_CSV_FILTERED}")

In [ ]:
#feature seleciton on everything 

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
file_path = os.path.join(PROJECT_ROOT, "echtfinal", "vettedcellcounts_2april_all.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "echtfinal", "UMAP_2april")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
# Features are identified by checking if the column name is purely numeric
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# 3.5 FEATURE SUMMARY PRINT-OUT
print("\n" + "="*45)
print(f"{'Channel Selection':<25} | {'Features Found':<15}")
print("-" * 45)
for config in plot_configs:
    print(f"{config['name']:<25} | {len(config['indices']):<15}")
print("="*45 + "\n")

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    feature_count = len(config['indices'])
    print(f"Processing interactive UMAP for: {config['name']} ({feature_count} features)...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    
    # UMAP parameters
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        hover_data={
            'Plate': True, 
            'Well_ID': True, 
            'Cell_Count': True, 
            'UMAP1': False, 
            'UMAP2': False
        },
        title=f"UMAP: {config['name']} ({feature_count} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) 
    fig.update_traces(marker=dict(size=9, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square')) 
    
    fig.update_layout(
        width=850, 
        height=850,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Save the interactive HTML
    save_name = f"UMAP5_{config['name'].replace(' ', '_')}.html"
    save_path = os.path.join(OUTPUT_DIR, save_name)
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    
    # Show the plot
    fig.show()

In [ ]:
#plotting all

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
INPUT_CSV = os.path.join(PROJECT_ROOT, "echtfinal", "aggregated_wells_median_min5.csv") 
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "echtfinal")
CONTROL_LABEL = "no_sgRNA" 

print("Loading raw data...")
df_raw = pd.read_csv(INPUT_CSV)
df_raw.columns = [str(c) for c in df_raw.columns]

# Separate Metadata and Features
metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count']
feature_cols = [c for c in df_raw.columns if c not in metadata_cols]

# ==========================================
# TOOLBOX: FUNCTIONS
# ==========================================

def filter_within_plate_consistency(df, features, top_n_to_keep=2000):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

def filter_across_plate_stability(df, features, top_n_to_keep=200):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_batch_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    if not tp_batch_noises:
        return features 
        
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    
    # Selection based on batch noise ranking
    stable_features = total_batch_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

def filter_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

# ==========================================
# EXECUTION PIPELINE
# ==========================================

# --- STEP 0: GLOBAL VARIANCE FILTER ---
# Removes features that are flat/near-zero across the whole experiment first
print(f"\nRunning Step 0: Global Variance Filter (std > 0.01)...")
initial_std = df_raw[feature_cols].std()
active_features = initial_std[initial_std > 0.01].index.tolist()
print(f"Removed {len(feature_cols) - len(active_features)} low-variance features.")

# --- STEP 1: WITHIN-PLATE CONSISTENCY ---
print(f"Running Step 1: Within-Plate Consistency (Filtering to top 2000)...")
step1_features = filter_within_plate_consistency(df_raw, active_features, top_n_to_keep=2000)
df_step1 = df_raw[metadata_cols + step1_features]

# --- STEP 2: ACROSS-PLATE STABILITY ---
print(f"Running Step 2: Across-Plate Stability (Filtering to top 200)...")
step2_features = filter_across_plate_stability(df_step1, step1_features, top_n_to_keep=200)
df_step2 = df_step1[metadata_cols + step2_features]

# --- STEP 3: REDUNDANCY REMOVAL ---
print(f"Running Step 3: Redundancy Filter (Threshold 0.9)...")
final_feature_list = filter_redundancy(df_step2, step2_features, correlation_threshold=0.9)

# ==========================================
# FINAL SAVE
# ==========================================
df_final = df_step2[metadata_cols + final_feature_list]
output_path = os.path.join(OUTPUT_DIR, "vettedcellcounts_2april_all.csv")
df_final.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE")
print(f"Original features: {len(feature_cols)}")
print(f"Active features (Step 0): {len(active_features)}")
print(f"Final feature count: {len(final_feature_list)}")
print(f"Saved to: {output_path}")
print("="*40)

In [ ]:
#time overlay

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import plotly.express as px

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
file_path = os.path.join(PROJECT_ROOT, "2april", "vettedcellcounts_2april_all.csv")
df = pd.read_csv(file_path)

HTML_DIR = os.path.join(PROJECT_ROOT, "2april", "UMAP_Time_all")
SVG_DIR = os.path.join(PROJECT_ROOT, "2april", "UMAP_Time_all")

for folder in [HTML_DIR, SVG_DIR]:
    if not os.path.exists(folder):
        os.makedirs(folder)

df.columns = [str(c) for c in df.columns]
df['Timepoint'] = df['Plate'].str.extract(r'(T\d+)')
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if str(x).lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# --- 2. COLOR SCHEMES ---
unique_combos = sorted(df['Plate'].unique())
turbo_colors = px.colors.sample_colorscale("Turbo", [i/(len(unique_combos)-1) for i in range(len(unique_combos))])
combo_color_map = {combo: turbo_colors[i] for i, combo in enumerate(unique_combos)}

time_colors = {'T0': '#A9D1FF', 'T1': '#2A7FFF', 'T2': "#1647AA"}

# --- 3. CHANNEL SELECTION ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# --- 4. EXECUTION ---
for config in plot_configs:
    if not config['indices']: continue
    print(f"Generating Plots and SVGs for: {config['name']}...")
    
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df_plot = df.copy()
    df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]

    modes = [
        ('PLATE', unique_combos, combo_color_map, 'Plate-Time View'),
        ('TIME', sorted(df_plot['Timepoint'].unique()), time_colors, 'Timepoint View')
    ]

    for mode_name, groups, color_map, title_prefix in modes:
        fig = go.Figure()
        
        for group in groups:
            for t_type in ['Mutant', 'Control']:
                mask = (df_plot['Plate' if mode_name == 'PLATE' else 'Timepoint'] == group) & (df_plot['Type'] == t_type)
                curr = df_plot[mask]
                if curr.empty: continue
                
                color = color_map[group]
                
                fig.add_trace(go.Scatter(
                    x=curr['UMAP1'], y=curr['UMAP2'], mode='markers',
                    name=str(group),
                    marker=dict(
                        color=color, 
                        size=8 if t_type == 'Mutant' else 11,
                        symbol='circle' if t_type == 'Mutant' else 'square',
                        line=dict(width=1.0, color='black') if t_type == 'Control' else dict(width=0),
                        opacity=1.0
                    ),
                    customdata=np.stack((curr['Plate'], curr['Well_ID'], curr['Treatment'], curr['Cell_Count']), axis=-1),
                    hovertemplate="<b>%{customdata[2]}</b><br>Plate: %{customdata[0]}<br>Well: %{customdata[1]}<br>Count: %{customdata[3]}<extra></extra>",
                    showlegend=True if t_type == 'Mutant' else False,
                    legendgroup=str(group)
                ))

        # --- UPDATED LAYOUT SECTION ---
        fig.update_layout(
            title=f"{title_prefix}: {config['name']}",
            template='plotly_white',
            width=850, height=850,
            xaxis=dict(
                title="UMAP 1",
                showgrid=False,
                showticklabels=True,  # Show Axis Numbers
                showline=True,        # Show Axis Border
                linecolor='black',
                zeroline=False
            ),
            yaxis=dict(
                title="UMAP 2",
                showgrid=False,
                showticklabels=True,  # Show Axis Numbers
                showline=True,        # Show Axis Border
                linecolor='black',
                zeroline=False
            )
        )

        html_name = f"{config['name'].replace(' ', '_')}_{mode_name}.html"
        fig.write_html(os.path.join(HTML_DIR, html_name))
        
        svg_name = f"{config['name'].replace(' ', '_')}_{mode_name}.svg"
        fig.write_image(os.path.join(SVG_DIR, svg_name))

    print(f"Successfully saved {config['name']} exports.")

print("\nAll files (HTML and SVG) are ready.")

In [ ]:
#with annotation

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "2april", "UMAP_all_annotated_nonames")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

file_path = os.path.join(PROJECT_ROOT, "2april", "vettedcellcounts_2april_all.csv")
anno_path = os.path.join(PROJECT_ROOT, "Pathway_annotation.xlsx")

print("Loading data...")
df_raw = pd.read_csv(file_path)
anno_df = pd.read_excel(anno_path)
df_raw.columns = [str(c) for c in df_raw.columns]

# --- 2. GLOBAL VARIANCE FILTER ---
metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count']
feature_cols = [c for c in df_raw.columns if c.isdigit()]
initial_std = df_raw[feature_cols].std()
active_features = initial_std[initial_std > 0.01].index.tolist()
df = df_raw[metadata_cols + active_features].copy()

# --- 3. MERGE & CATEGORIZATION ---
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')
df['Effective_Annotation'] = df['SubtiWiki Annotation 4'].fillna(df['SubtiWiki Annotation 3']).fillna("Unknown/Other")
df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

is_control = df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
df['Display_Category'] = df['Effective_Annotation']
df.loc[is_control, 'Display_Category'] = 'no_sgrna'
df.loc[(df['Timepoint'] == 'T0') & (~is_control), 'Display_Category'] = 'Baseline (T0)'

# --- 4. GOLDEN ANGLE COLOR MAPPING ---
# This ensures maximum perceptual distance between consecutive categories
all_cats = sorted([c for c in df['Display_Category'].unique() if c not in ['no_sgrna', 'Baseline (T0)', 'Unknown/Other']])
num_cats = len(all_cats)

if num_cats > 0:
    # We sample 256 colors from a high-quality spectrum
    base_palette = px.colors.sample_colorscale("Turbo", [i/255 for i in range(256)])
    
    # Use the Golden Angle (~137.5 degrees converted to index steps) 
    # to jump through the palette so neighbors are always far apart
    golden_ratio_conjugate = 0.618033988749895
    h_values = [(i * golden_ratio_conjugate) % 1 for i in range(num_cats)]
    
    # Map these "jumps" to the palette
    max_contrast_palette = [base_palette[int(h * 255)] for h in h_values]
    
    color_map = {cat: max_contrast_palette[i] for i, cat in enumerate(all_cats)}
else:
    color_map = {}

color_map['no_sgrna'] = '#EBEBEB'
color_map['Baseline (T0)'] = '#B0B0B0'
color_map['Unknown/Other'] = '#222222'

# --- 5. EXECUTION ---
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Vetted_Channels", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel_2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel_3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel_4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel_5", "indices": get_channel_features(5120, 6400)}
]

for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Processing: {config['name']}...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    df['UMAP1'], df['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    df['Point_Shape'] = df['Timepoint'].map({"T0": "circle", "T1": "x", "T2": "circle"})
    df.loc[is_control, 'Point_Shape'] = 'square'

    fig = px.scatter(
        df, x='UMAP1', y='UMAP2', 
        color='Display_Category',
        symbol='Point_Shape',
        symbol_map={"circle": "circle", "x": "x", "square": "square"},
        hover_name='Treatment',
        hover_data=['Plate', 'Effective_Annotation'],
        title=f"Pathway Overlay: {config['name'].replace('_', ' ')}",
        color_discrete_map=color_map,
        template='plotly_white'
    )
    
    seen_pathways = set()
    fig.for_each_trace(lambda t: (
        t.update(showlegend=False) if t.name.split(",")[0] in seen_pathways 
        else (seen_pathways.add(t.name.split(",")[0]), t.update(name=t.name.split(",")[0]))
    ))

    # Slightly higher opacity and black outlines for points to make colors pop
    fig.update_traces(marker=dict(opacity=0.9, line=dict(width=0.5, color='white'))) 
    fig.update_traces(marker=dict(size=5), selector=dict(marker_symbol='circle'))
    fig.update_traces(marker=dict(size=7), selector=dict(marker_symbol='x'))
    fig.update_traces(marker=dict(size=10, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square'))
    
    fig.update_layout(
        width=1400, height=900,
        legend_title_text='Pathway / Group',
        xaxis=dict(title="UMAP 1", showline=True, linewidth=2, linecolor='black', showgrid=False),
        yaxis=dict(title="UMAP 2", showline=True, linewidth=2, linecolor='black', showgrid=False)
    )
    
    file_base = f"UMAP_Annotated_{config['name']}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{file_base}.html"))

print(f"Done. Golden-angle sampling applied for maximum distinction.")

In [ ]:
#enkelT0enT1 bekijken
###
#

In [ ]:
#preselection T0 en T1 enkel 

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
INPUT_CSV = os.path.join(PROJECT_ROOT, "echtfinal", "aggregated_wells_median_min5.csv") 
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "echtfinal")
CONTROL_LABEL = "no_sgRNA" 

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

print("Loading raw data...")
df_raw_full = pd.read_csv(INPUT_CSV)
df_raw_full.columns = [str(c) for c in df_raw_full.columns]

# --- NEW STEP: FILTER OUT T2 AND SAVE COPY ---
print("Filtering out T2 samples...")
# This assumes your Plate column ends with the timepoint (e.g., 'PlateName_T0')
df_raw = df_raw_full[df_raw_full['Plate'].str.contains('T0|T1')].copy()

# Save the T0_T1 raw subset
filtered_raw_path = os.path.join(OUTPUT_DIR, "raw_data_T0_T1_only.csv")
df_raw.to_csv(filtered_raw_path, index=False)
print(f"Saved filtered raw copy to: {filtered_raw_path}")

# Separate Metadata and Features
metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count']
feature_cols = [c for c in df_raw.columns if c not in metadata_cols]

# ==========================================
# TOOLBOX: FUNCTIONS
# ==========================================

def filter_within_plate_consistency(df, features, top_n_to_keep=2000):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

def filter_across_plate_stability(df, features, top_n_to_keep=200):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_batch_noises = []
    # UPDATED: Removed 'T2' from the loop
    for tp in ['T0', 'T1']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    if not tp_batch_noises:
        print("Warning: Not enough plates per timepoint to calculate stability. Returning input features.")
        return features 
        
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    stable_features = total_batch_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

def filter_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

# ==========================================
# EXECUTION PIPELINE
# ==========================================

# --- STEP 0: GLOBAL VARIANCE FILTER ---
print(f"\nRunning Step 0: Global Variance Filter (std > 0.01)...")
initial_std = df_raw[feature_cols].std()
active_features = initial_std[initial_std > 0.01].index.tolist()
print(f"Removed {len(feature_cols) - len(active_features)} low-variance features.")

# --- STEP 1: WITHIN-PLATE CONSISTENCY ---
print(f"Running Step 1: Within-Plate Consistency (Filtering to top 2000)...")
step1_features = filter_within_plate_consistency(df_raw, active_features, top_n_to_keep=2000)
df_step1 = df_raw[metadata_cols + step1_features]

# --- STEP 2: ACROSS-PLATE STABILITY ---
print(f"Running Step 2: Across-Plate Stability (Filtering to top 200)...")
step2_features = filter_across_plate_stability(df_step1, step1_features, top_n_to_keep=200)
df_step2 = df_step1[metadata_cols + step2_features]

# --- STEP 3: REDUNDANCY REMOVAL ---
print(f"Running Step 3: Redundancy Filter (Threshold 0.9)...")
final_feature_list = filter_redundancy(df_step2, step2_features, correlation_threshold=0.9)

# ==========================================
# FINAL SAVE
# ==========================================
df_final = df_step2[metadata_cols + final_feature_list]
output_path = os.path.join(OUTPUT_DIR, "vettedcellcounts_2april_T0_T1.csv")
df_final.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE (T0 & T1 ONLY)")
print(f"Original rows (including T2): {len(df_raw_full)}")
print(f"Filtered rows (T0 & T1): {len(df_raw)}")
print(f"Final feature count: {len(final_feature_list)}")
print(f"Saved results to: {output_path}")
print("="*40)

In [ ]:
#plot enkel T0 en T1

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
file_path = os.path.join(PROJECT_ROOT, "2april", "vettedcellcounts_2april_T0_T1.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "2april", "UMAP_2april_T0T1")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE2_T0","PLATE2_T1","PLATE3_T0","PLATE3_T1",
                   "PLATE4_T0","PLATE4_T1","PLATE5_T0","PLATE5_T1"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
# Features are identified by checking if the column name is purely numeric
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# 3.5 FEATURE SUMMARY PRINT-OUT
print("\n" + "="*45)
print(f"{'Channel Selection':<25} | {'Features Found':<15}")
print("-" * 45)
for config in plot_configs:
    print(f"{config['name']:<25} | {len(config['indices']):<15}")
print("="*45 + "\n")

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    feature_count = len(config['indices'])
    print(f"Processing interactive UMAP for: {config['name']} ({feature_count} features)...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    
    # UMAP parameters
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        hover_data={
            'Plate': True, 
            'Well_ID': True, 
            'Cell_Count': True, 
            'UMAP1': False, 
            'UMAP2': False
        },
        title=f"UMAP: {config['name']} ({feature_count} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) 
    fig.update_traces(marker=dict(size=9, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square')) 
    
    fig.update_layout(
        width=850, 
        height=850,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Save the interactive HTML
    save_name = f"UMAP5_{config['name'].replace(' ', '_')}.html"
    save_path = os.path.join(OUTPUT_DIR, save_name)
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    
    # Show the plot
    fig.show()

In [ ]:
#time overlay T0 en T1

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import plotly.express as px

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
file_path = os.path.join(PROJECT_ROOT, "2april", "vettedcellcounts_2april_T0_T1.csv")
df = pd.read_csv(file_path)

HTML_DIR = os.path.join(PROJECT_ROOT, "2april", "UMAP_Time_T0T1")
SVG_DIR = os.path.join(PROJECT_ROOT, "2april", "UMAP_Time_T0T1")

for folder in [HTML_DIR, SVG_DIR]:
    if not os.path.exists(folder):
        os.makedirs(folder)

df.columns = [str(c) for c in df.columns]
df['Timepoint'] = df['Plate'].str.extract(r'(T\d+)')
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if str(x).lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# --- 2. COLOR SCHEMES ---
unique_combos = sorted(df['Plate'].unique())
turbo_colors = px.colors.sample_colorscale("Turbo", [i/(len(unique_combos)-1) for i in range(len(unique_combos))])
combo_color_map = {combo: turbo_colors[i] for i, combo in enumerate(unique_combos)}

time_colors = {'T0': '#A9D1FF', 'T1': '#2A7FFF', 'T2': "#1647AA"}

# --- 3. CHANNEL SELECTION ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# --- 4. EXECUTION ---
for config in plot_configs:
    if not config['indices']: continue
    print(f"Generating Plots and SVGs for: {config['name']}...")
    
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df_plot = df.copy()
    df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]

    modes = [
        ('PLATE', unique_combos, combo_color_map, 'Plate-Time View'),
        ('TIME', sorted(df_plot['Timepoint'].unique()), time_colors, 'Timepoint View')
    ]

    for mode_name, groups, color_map, title_prefix in modes:
        fig = go.Figure()
        
        for group in groups:
            for t_type in ['Mutant', 'Control']:
                mask = (df_plot['Plate' if mode_name == 'PLATE' else 'Timepoint'] == group) & (df_plot['Type'] == t_type)
                curr = df_plot[mask]
                if curr.empty: continue
                
                color = color_map[group]
                
                fig.add_trace(go.Scatter(
                    x=curr['UMAP1'], y=curr['UMAP2'], mode='markers',
                    name=str(group),
                    marker=dict(
                        color=color, 
                        size=8 if t_type == 'Mutant' else 11,
                        symbol='circle' if t_type == 'Mutant' else 'square',
                        line=dict(width=1.0, color='black') if t_type == 'Control' else dict(width=0),
                        opacity=1.0
                    ),
                    customdata=np.stack((curr['Plate'], curr['Well_ID'], curr['Treatment'], curr['Cell_Count']), axis=-1),
                    hovertemplate="<b>%{customdata[2]}</b><br>Plate: %{customdata[0]}<br>Well: %{customdata[1]}<br>Count: %{customdata[3]}<extra></extra>",
                    showlegend=True if t_type == 'Mutant' else False,
                    legendgroup=str(group)
                ))

        # --- UPDATED LAYOUT SECTION ---
        fig.update_layout(
            title=f"{title_prefix}: {config['name']}",
            template='plotly_white',
            width=850, height=850,
            xaxis=dict(
                title="UMAP 1",
                showgrid=False,
                showticklabels=True,  # Show Axis Numbers
                showline=True,        # Show Axis Border
                linecolor='black',
                zeroline=False
            ),
            yaxis=dict(
                title="UMAP 2",
                showgrid=False,
                showticklabels=True,  # Show Axis Numbers
                showline=True,        # Show Axis Border
                linecolor='black',
                zeroline=False
            )
        )

        html_name = f"{config['name'].replace(' ', '_')}_{mode_name}.html"
        fig.write_html(os.path.join(HTML_DIR, html_name))
        
        svg_name = f"{config['name'].replace(' ', '_')}_{mode_name}.svg"
        fig.write_image(os.path.join(SVG_DIR, svg_name))

    print(f"Successfully saved {config['name']} exports.")

print("\nAll files (HTML and SVG) are ready.")

In [ ]:
#annotation T0T1, met nieuwe kleur

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import plotly.express as px

# --- 1. SETUP & THE COLOR DICTIONARY ---
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "echtfinal", "UMAP_Strict_Manual")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# EXPLICIT COLOR MAP - This is the "Source of Truth"
MANUAL_COLORS = {
    "biosynthesis of peptidoglycan": "#1f77b4", 
    "biosynthesis of teichoic acid": "#ff7f0e", 
    "aminoacyl-tRNA synthetases": "#2ca02c",     
    "cell shape": "#d62728",                     
    "biosynthesis of fatty acids": "#9467bd",    
    "DNA replication": "#8c564b",                
    "DNA condensation/ segregation": "#e377c2",  
    "biosynthesis of isoprenoids": "#7f7f7f",    
    "cell division": "#bcbd22",                  
    "ribosomal proteins": "#17becf",             
    "biosynthesis of iron-sulfur clusters": "#aec7e8", 
    "glycolysis": "#ffbb78",                     
    "biosynthesis of menaquinone": "#98df8a",
    "Control Group": "#333333",  # Charcoal
    "Baseline (T0)": "#808080",  # Medium Grey
    "Unknown/Other": "#D3D3D3"   # Light Grey
}

# --- 2. DATA LOADING ---
file_path = os.path.join(PROJECT_ROOT, "echtfinal", "vettedcellcounts_2april_T0_T1.csv")
anno_path = os.path.join(PROJECT_ROOT, "Pathway_annotation.xlsx")

df_raw = pd.read_csv(file_path)
anno_df = pd.read_excel(anno_path)
df_raw.columns = [str(c) for c in df_raw.columns]

df = df_raw.copy()
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')

# Clean up strings to prevent "Glycolysis " vs "Glycolysis" errors
df['SubtiWiki Annotation 4'] = df['SubtiWiki Annotation 4'].astype(str).str.strip()
df['SubtiWiki Annotation 3'] = df['SubtiWiki Annotation 3'].astype(str).str.strip()

df['Effective_Annotation'] = df['SubtiWiki Annotation 4'].replace('nan', np.nan).fillna(df['SubtiWiki Annotation 3'].replace('nan', np.nan)).fillna("Unknown/Other")
df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

is_ctrl = df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
df['Display_Category'] = df['Effective_Annotation']
df.loc[(df['Timepoint'] == 'T0') & (~is_ctrl), 'Display_Category'] = 'Baseline (T0)'
df.loc[is_ctrl, 'Display_Category'] = 'Control Group'

# --- 3. FINAL COLOR MAPPING & ORDER ---
unique_in_data = df['Display_Category'].unique()
other_cats = sorted([c for c in unique_in_data if c not in MANUAL_COLORS])

# Legend Order
final_legend_order = ["Control Group", "Baseline (T0)"] + list(MANUAL_COLORS.keys())[0:13] + other_cats + ["Unknown/Other"]

# Fill in colors for "others" using a backup palette
auto_pal = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24
color_lookup = MANUAL_COLORS.copy()
for i, cat in enumerate(other_cats):
    color_lookup[cat] = auto_pal[i % len(auto_pal)]

# --- 4. UMAP ---
feature_cols = [c for c in df_raw.columns if c.isdigit()]
X_scaled = StandardScaler().fit_transform(df[feature_cols].values)
embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
df['UMAP1'], df['UMAP2'] = embedding[:, 0], embedding[:, 1]

# --- 5. PLOTTING ---
fig = go.Figure()

# Drawing order reversed: background first, priority/controls last (so they are on top)
for cat in final_legend_order[::-1]:
    if cat not in df['Display_Category'].values:
        continue
    
    cat_df = df[df['Display_Category'] == cat]
    
    # Identify Shapes
    is_c_cat = (cat == "Control Group")
    
    # We create specific groups for Mutants, T0 Controls, T1 Controls
    sub_groups = [
        ('circle', cat_df[~cat_df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])]),
        ('square', cat_df[(cat_df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])) & (cat_df['Timepoint'] == 'T0')]),
        ('diamond', cat_df[(cat_df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])) & (cat_df['Timepoint'] == 'T1')])
    ]
    
    for sym, sub_df in sub_groups:
        if sub_df.empty: continue
        
        # Hard-force size: 10 for Controls, 6 for others
        sz = 9 if is_c_cat else 8
        # Hard-force color: pull from color_lookup
        clr = color_lookup.get(cat, "#000000")

        fig.add_trace(go.Scatter(
            x=sub_df['UMAP1'], y=sub_df['UMAP2'],
            mode='markers',
            name=cat,
            legendgroup=cat,
            showlegend=False, 
            marker=dict(
                size=sz,
                color=clr,  # THIS FORCES YOUR HEX COLOR
                symbol=sym,
                opacity=0.9 if is_c_cat else 0.8,
                line=dict(width=0.4, color='white')
            ),
            text=[f"<b>{cat}</b><br>Tr: {r['Treatment']}<br>Tp: {r['Timepoint']}" for _, r in sub_df.iterrows()],
            hoverinfo='text'
        ))

# --- 6. LEGEND ORDER OVERRIDE ---
seen = set()
for cat in final_legend_order:
    for trace in fig.data:
        if trace.name == cat and cat not in seen:
            trace.showlegend = True
            seen.add(cat)
            trace.legendrank = final_legend_order.index(cat)

# --- 7. LAYOUT ---
dim = 900
fig.update_layout(
    template='plotly_white',
    width=dim + 450, height=dim,
    margin=dict(l=60, r=20, b=60, t=80),
    legend=dict(title_text='<b>Pathways</b>', x=1.02, y=1, font=dict(size=10)),
    xaxis=dict(title="UMAP 1", showline=True, linecolor='black', showgrid=False, scaleanchor="y", scaleratio=1),
    yaxis=dict(title="UMAP 2", showline=True, linecolor='black', showgrid=False)
)

fig.write_html(os.path.join(OUTPUT_DIR, "UMAP_IronClad_Colors.html"))
print("Done. Colors are hard-coded per point and layered correctly.")

In [ ]:
#normale contour

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import plotly.express as px

# --- 1. SETUP & THE COLOR DICTIONARY ---
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "echtfinal", "UMAP_Strict_ManualContourwerkend")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# EXPLICIT COLOR MAP
MANUAL_COLORS = {
    "biosynthesis of peptidoglycan": "#1f77b4", 
    "biosynthesis of teichoic acid": "#ff7f0e", 
    "aminoacyl-tRNA synthetases": "#2ca02c",     
    "cell shape": "#d62728",                       
    "biosynthesis of fatty acids": "#9467bd",    
    "DNA replication": "#8c564b",                 
    "DNA condensation/ segregation": "#e377c2",  
    "biosynthesis of isoprenoids": "#7f7f7f",    
    "cell division": "#bcbd22",                   
    "ribosomal proteins": "#17becf",             
    "biosynthesis of iron-sulfur clusters": "#aec7e8", 
    "glycolysis": "#ffbb78",                       
    "biosynthesis of menaquinone": "#98df8a",
    "Control Group": "#000000",  # Set to Black for the legend, contours are black
    "Baseline (T0)": "#808080",  # Medium Grey
    "Unknown/Other": "#D3D3D3"   # Light Grey
}

# --- 2. DATA LOADING ---
file_path = os.path.join(PROJECT_ROOT, "echtfinal", "vettedcellcounts_2april_T0_T1.csv")
anno_path = os.path.join(PROJECT_ROOT, "Pathway_annotation.xlsx")

df_raw = pd.read_csv(file_path)
anno_df = pd.read_excel(anno_path)
df_raw.columns = [str(c) for c in df_raw.columns]

df = df_raw.copy()
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')

df['SubtiWiki Annotation 4'] = df['SubtiWiki Annotation 4'].astype(str).str.strip()
df['SubtiWiki Annotation 3'] = df['SubtiWiki Annotation 3'].astype(str).str.strip()

df['Effective_Annotation'] = df['SubtiWiki Annotation 4'].replace('nan', np.nan).fillna(df['SubtiWiki Annotation 3'].replace('nan', np.nan)).fillna("Unknown/Other")
df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

is_ctrl = df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
df['Display_Category'] = df['Effective_Annotation']
# NON-CONTROL T0 are marked as Baseline (T0)
df.loc[(df['Timepoint'] == 'T0') & (~is_ctrl), 'Display_Category'] = 'Baseline (T0)'
# ANY CONTROL (T0, T1, T2) is marked as Control Group
df.loc[is_ctrl, 'Display_Category'] = 'Control Group'

# --- 3. FINAL COLOR MAPPING & ORDER ---
unique_in_data = df['Display_Category'].unique()
other_cats = sorted([c for c in unique_in_data if c not in MANUAL_COLORS])

# Legend Order
final_legend_order = ["Control Group", "Baseline (T0)"] + list(MANUAL_COLORS.keys())[0:13] + other_cats + ["Unknown/Other"]

# Fill in colors for "others" using a backup palette
auto_pal = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24
color_lookup = MANUAL_COLORS.copy()
for i, cat in enumerate(other_cats):
    color_lookup[cat] = auto_pal[i % len(auto_pal)]

# --- 4. UMAP ---
feature_cols = [c for c in df_raw.columns if c.isdigit()]
X_scaled = StandardScaler().fit_transform(df[feature_cols].values)
embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
df['UMAP1'], df['UMAP2'] = embedding[:, 0], embedding[:, 1]

# --- 5. PLOTTING ---
fig = go.Figure()

# 5.1 First, plot the Control Group as Density Contours (Black lines, no fill)
ctrl_df = df[df['Display_Category'] == 'Control Group']
if not ctrl_df.empty:
    fig.add_trace(go.Histogram2dContour(
        x=ctrl_df['UMAP1'],
        y=ctrl_df['UMAP2'],
        name='Control Group',
        legendgroup='Control Group',
        showlegend=True,
        contours=dict(
            coloring='none',      # Removes the color fill/gradient
            showlines=True,       # Draws only the lines
            showlabels=False      # Removes the number labels
        ),
        line=dict(width=1.2, color='black'),
        ncontours=12,             # Number of contour lines
        hoverinfo='skip'
    ))

# 5.2 Plot all other categories as markers (Circles)
# Filter out "Control Group" from the loop
remaining_cats = [c for c in final_legend_order[::-1] if c != "Control Group"]

for cat in remaining_cats:
    if cat not in df['Display_Category'].values:
        continue
    
    sub_df = df[df['Display_Category'] == cat]
    
    clr = color_lookup.get(cat, "#000000")

    fig.add_trace(go.Scatter(
        x=sub_df['UMAP1'], y=sub_df['UMAP2'],
        mode='markers',
        name=cat,
        legendgroup=cat,
        showlegend=False, 
        marker=dict(
            size=8,
            color=clr,
            symbol='circle',     # ALL markers are now circles
            opacity=0.8,
            line=dict(width=0.4, color='white')
        ),
        text=[f"<b>{cat}</b><br>Tr: {r['Treatment']}<br>Tp: {r['Timepoint']}" for _, r in sub_df.iterrows()],
        hoverinfo='text'
    ))

# --- 6. LEGEND ORDER OVERRIDE ---
seen = set()
for cat in final_legend_order:
    for trace in fig.data:
        # Check if trace name matches AND it's the first time seeing this category
        if trace.name == cat and cat not in seen:
            trace.showlegend = True
            seen.add(cat)
            trace.legendrank = final_legend_order.index(cat)

# --- 7. LAYOUT ---
dim = 900
fig.update_layout(
    template='plotly_white',
    width=dim + 450, height=dim,
    margin=dict(l=60, r=20, b=60, t=80),
    legend=dict(title_text='<b>Pathways</b>', x=1.02, y=1, font=dict(size=10)),
    xaxis=dict(title="UMAP 1", showline=True, linecolor='black', showgrid=False, scaleanchor="y", scaleratio=1),
    yaxis=dict(title="UMAP 2", showline=True, linecolor='black', showgrid=False)
)

fig.write_html(os.path.join(OUTPUT_DIR, "UMAP_Density_Controls_GreyDots_Baseline.html"))
print(f"Plot complete. Check: {os.path.join(OUTPUT_DIR, 'UMAP_Density_Controls_GreyDots_Baseline.html')}")

In [ ]:
#area overlay

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go

# ==========================================
# 1. SETUP & DATA MAPPING
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'

file_path = os.path.join(PROJECT_ROOT, "2april", "vettedcellcounts_2april_T0_T1.csv")
df = pd.read_csv(file_path)

df.columns = [str(c) for c in df.columns]

# --- LOAD AUC DATA WITH YOUR PATHS ---
auc_no_xylose_path = r'C:\Users\arnou\Documents\thesis\Resultaten\GrowthCurves\AUC_without_xylose.csv' 
auc_with_xylose_path = r'C:\Users\arnou\Documents\thesis\Resultaten\GrowthCurves\AUC_with_xylose.csv'

auc_no_xylose = pd.read_csv(auc_no_xylose_path)
auc_with_xylose = pd.read_csv(auc_with_xylose_path)

map_no_xylose = dict(zip(auc_no_xylose['Gene_target'], auc_no_xylose['AUC']))
map_with_xylose = dict(zip(auc_with_xylose['Gene_target'], auc_with_xylose['AUC']))

def assign_auc(row):
    treatment = str(row['Treatment'])
    plate = str(row['Plate'])
    # Return NaN if missing to keep it out of the color scale
    if "_T0" in plate:
        return map_no_xylose.get(treatment, np.nan)
    else:
        return map_with_xylose.get(treatment, np.nan)

# Create AUC column (NaNs preserved for grey coloring)
df['AUC'] = df.apply(assign_auc, axis=1)

# Identify 'Type' for symbols
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if str(x).lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "2april", "UMAP_2april_AUC")
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# ==========================================
# 2. FEATURE SELECTION
# ==========================================
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# ==========================================
# 3. RUN UMAP & PLOT
# ==========================================
for config in plot_configs:
    if not config['indices']:
        continue
    
    print(f"Processing UMAP for: {config['name']}...")
    
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Split data to protect the color scale range
    df_valid = df_plot[df_plot['AUC'].notna()]
    df_missing = df_plot[df_plot['AUC'].isna()]
    
    # Trace 1: Valid AUC (Viridis Scale)
    fig = px.scatter(
        df_valid, 
        x='UMAP1', 
        y='UMAP2', 
        color='AUC',
        symbol='Type',
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'AUC': ':.4f', 'UMAP1': False, 'UMAP2': False},
        color_continuous_scale='Viridis',
        title=f"UMAP: {config['name']} (Grey = Missing AUC)",
        template='plotly_white'
    )
    
    # Trace 2: Missing AUC (Static Grey)
    fig.add_trace(
        go.Scatter(
            x=df_missing['UMAP1'],
            y=df_missing['UMAP2'],
            mode='markers',
            marker=dict(color='lightgrey', size=8, opacity=0.5, line=dict(width=0.5, color='DarkGrey')),
            name='No AUC Data',
            text=df_missing['Treatment'],
            customdata=np.stack((df_missing['Plate'], df_missing['Well_ID']), axis=-1),
            hovertemplate="<b>%{text}</b><br>Plate: %{customdata[0]}<br>Well: %{customdata[1]}<br>AUC: N/A<extra></extra>"
        )
    )
    
    fig.update_traces(marker=dict(size=8, opacity=0.8, line=dict(width=0.5, color='DarkGrey')), selector=dict(mode='markers'))
    
    fig.update_layout(
        width=950, height=850,
        coloraxis_colorbar=dict(title="AUC Score"),
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    save_name = f"UMAP_AUC_{config['name'].replace(' ', '_')}.html"
    save_path = os.path.join(OUTPUT_DIR, save_name)
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    fig.show()

In [ ]:
#mask overlay

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# ==========================================
# 1. SETUP & DATA MAPPING
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
file_path = os.path.join(PROJECT_ROOT, "2april", "vettedcellcounts_2april_T0_T1.csv")
df = pd.read_csv(file_path)

# Ensure column names are strings for consistency
df.columns = [str(c) for c in df.columns]

# Map Area Data
area_path = r'D:\Thesis\final\area\master_median_areas_per_site.csv'
area_df = pd.read_csv(area_path)
area_df['Well_ID'] = area_df['Plate'] + "_" + area_df['Well']
area_map = area_df.groupby('Well_ID')['Median_Area'].median().to_dict()

# Add Area to main dataframe and handle missing values
df['Median_Area_Size'] = df['Well_ID'].map(area_map)
df['Median_Area_Size'] = df['Median_Area_Size'].fillna(df['Median_Area_Size'].mean())

# Identify 'Type' for symbols
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "2april", "UMAP_2april_maskarea")
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# ==========================================
# 2. FEATURE SELECTION
# ==========================================
vetted_features = [f for f in df.columns if f.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# ==========================================
# 3. RUN UMAP & PLOT WITH ROBUST COLOR SCALE
# ==========================================
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing UMAP for: {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # --- ROBUST COLOR SCALING ---
    # We find the 95th percentile to prevent outliers from squashing the gradient
    color_min = df_plot['Median_Area_Size'].min()
    color_max = df_plot['Median_Area_Size'].quantile(0.99) 
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Median_Area_Size',     
        symbol='Type',                
        hover_name='Treatment',
        hover_data={
            'Plate': True, 
            'Well_ID': True, 
            'Cell_Count': True,
            'Median_Area_Size': ':.2f',
            'UMAP1': False, 
            'UMAP2': False
        },
        range_color=[color_min, color_max], # This forces the gradient to ignore outliers
        color_continuous_scale='Viridis',
        title=f"UMAP: {config['name']} (Color Cap at 95th Percentile)",
        template='plotly_white'
    )
    
    # Adjust point appearance
    fig.update_traces(marker=dict(size=8, opacity=0.8, line=dict(width=0.5, color='DarkGrey')))
    
    # Clean up layout
    fig.update_layout(
        width=950, 
        height=850,
        coloraxis_colorbar=dict(
            title="Median Area",
            ticksuffix="+" if color_max < df_plot['Median_Area_Size'].max() else ""
        ),
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Save and Show
    save_name = f"UMAP_FIXED_GRADIENTcap_{config['name'].replace(' ', '_')}.html"
    save_path = os.path.join(OUTPUT_DIR, save_name)
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    fig.show()

In [ ]:
#antibiotics#
#######






#

In [ ]:
#aggregation antibiotics 

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from tqdm import tqdm

# ==========================================
# 1. SETUP & PATHS (MULTI-PLATE)
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
PLATES = ["PLATE6_T1","PLATE7_T1"]

# Paths for the three separate outputs
OUTPUT_CSV_MEDIAN = os.path.join(PROJECT_ROOT,"31march", "antibiotics_aggregated_wells_median.csv")
OUTPUT_CSV_MEAN   = os.path.join(PROJECT_ROOT,"31march", "antibiotics_aggregated_wells_mean.csv")
OUTPUT_CSV_STD    = os.path.join(PROJECT_ROOT,"31march", "antibiotics_aggregated_wells_std.csv")

CELL_COUNT_THRESHOLD = 0  
TREATMENT_COL = "Treatment"

all_plates_median = []
all_plates_mean = []
all_plates_std = []
all_cell_counts = [] 

for plate_id in PLATES:
    print(f"\n--- Processing {plate_id} ---")
    
    FEATURES_BASE = os.path.join(PROJECT_ROOT, "features", plate_id)
    METADATA_PATH = os.path.join(PROJECT_ROOT, "metadata", f"index_{plate_id}.csv")
    
    if not os.path.exists(METADATA_PATH):
        print(f"Skipping {plate_id}: Metadata not found.")
        continue

    meta = pd.read_csv(METADATA_PATH)
    well_storage = {}
    well_to_treatment = {}

    for i in tqdm(meta.index, desc=f"Loading {plate_id}"):
        well_id = f"{plate_id}_{meta.loc[i, 'Metadata_Well']}"
        treatment = str(meta.loc[i, TREATMENT_COL]).strip()
        
        filename = os.path.join(FEATURES_BASE, 
                                str(meta.loc[i, "Metadata_Well"]), 
                                f"{meta.loc[i, 'Metadata_Site']}.npz")
        
        if os.path.isfile(filename):
            try:
                with np.load(filename) as data:
                    cells = data["features"]
                    cells_f = cells[~np.isnan(cells).any(axis=1)]
                    
                    if len(cells_f) > 0:
                        if well_id not in well_storage:
                            well_storage[well_id] = []
                            well_to_treatment[well_id] = treatment
                        well_storage[well_id].append(cells_f)
            except:
                continue

    # --- AGGREGATION & THRESHOLDING STEP ---
    for well_id, feature_list in well_storage.items():
        all_cells_in_well = np.vstack(feature_list)
        well_cell_count = all_cells_in_well.shape[0]
        all_cell_counts.append(well_cell_count)

        if well_cell_count >= CELL_COUNT_THRESHOLD:
            # Calculate aggregations
            well_median = np.median(all_cells_in_well, axis=0)
            well_mean   = np.mean(all_cells_in_well, axis=0)
            well_std    = np.std(all_cells_in_well, axis=0)
            
            base_info = {
                "Plate": plate_id, 
                "Well_ID": well_id, 
                "Treatment": well_to_treatment[well_id],
                "Cell_Count": well_cell_count
            }
            
            # Efficiently map feature indices to values
            feat_cols = {idx: val for idx, val in enumerate(well_median)}
            all_plates_median.append({**base_info, **feat_cols})
            
            feat_cols_mean = {idx: val for idx, val in enumerate(well_mean)}
            all_plates_mean.append({**base_info, **feat_cols_mean})
            
            feat_cols_std = {idx: val for idx, val in enumerate(well_std)}
            all_plates_std.append({**base_info, **feat_cols_std})

# Convert to DataFrames
df_median = pd.DataFrame(all_plates_median)
df_mean   = pd.DataFrame(all_plates_mean)
df_std    = pd.DataFrame(all_plates_std)

# Helper function to reorder columns consistently
def reorder_cols(df):
    if df.empty: return df
    meta_cols = ["Plate", "Well_ID", "Treatment", "Cell_Count"]
    feat_cols = sorted([c for c in df.columns if c not in meta_cols])
    return df[meta_cols + feat_cols]

df_median = reorder_cols(df_median)
df_mean   = reorder_cols(df_mean)
df_std    = reorder_cols(df_std)

print(f"\nAggregation complete.")

# ==========================================
# 2. SAVE TO CSVs
# ==========================================
df_median.to_csv(OUTPUT_CSV_MEDIAN, index=False)
df_mean.to_csv(OUTPUT_CSV_MEAN, index=False)
df_std.to_csv(OUTPUT_CSV_STD, index=False)

print(f"Median data saved: {OUTPUT_CSV_MEDIAN}")
print(f"Mean data saved:   {OUTPUT_CSV_MEAN}")
print(f"Std Dev data saved: {OUTPUT_CSV_STD}")

In [ ]:
#min5 wells antiiobitcs (maar is niet nodig)

In [ ]:
import pandas as pd
import os

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
INPUT_DIR = os.path.join(PROJECT_ROOT, "31march")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "31march_filtered")

# Create the directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. DEFINE YOUR NEW THRESHOLD
STRICT_THRESHOLD = 5 

# List of the aggregation files you created in the previous step
FILES_TO_FILTER = [
    "antibiotics_aggregated_wells_median.csv",
    "antibiotics_aggregated_wells_mean.csv",
    "antibiotics_aggregated_wells_std.csv"
]

# 3. PROCESSING LOOP
for file_name in FILES_TO_FILTER:
    file_path = os.path.join(INPUT_DIR, file_name)
    
    if not os.path.exists(file_path):
        print(f"Skipping {file_name}: File not found.")
        continue

    print(f"\n--- Processing {file_name} ---")
    df = pd.read_csv(file_path)

    # Identify the wells to keep and remove
    # We use 'Cell_Count' which we added during the aggregation step
    filtered_df = df[df['Cell_Count'] >= STRICT_THRESHOLD].copy()
    removed_df = df[df['Cell_Count'] < STRICT_THRESHOLD].copy()

    # 4. REPORT
    print(f"Original wells: {len(df)}")
    print(f"Wells kept:     {len(filtered_df)}")
    print(f"Wells removed:  {len(removed_df)}")

    if not removed_df.empty and "median" in file_name:
        # Just show the list once (for the median file) to avoid clutter
        print(f"\nExample of removed wells (Count < {STRICT_THRESHOLD}):")
        print(removed_df[['Plate', 'Well_ID', 'Treatment', 'Cell_Count']].head(10).to_string(index=False))

    # 5. SAVE DATA
    # Renaming the output to include the 'min5' suffix
    output_name = file_name.replace(".csv", "_min5.csv")
    output_path = os.path.join(OUTPUT_DIR, output_name)
    filtered_df.to_csv(output_path, index=False)
    
    print(f"Filtered data saved to: {output_path}")

print("\nAll filtering tasks complete!")

In [ ]:
#zelf die paar slechte verwijderd in de juist ifle

In [ ]:
#puur mean plotten met antibiotica anotatie

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
file_path = os.path.join(PROJECT_ROOT, "31march_filtered", "antibiotics_aggregated_wells_mean_juist.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "2aprilAntibiotics", "UMAP_all_mean")
os.makedirs(OUTPUT_DIR, exist_ok=True)

df.columns = [str(c) for c in df.columns]

# 2. PRE-PROCESSING
# Extract Base Treatment (e.g., 'vancomycin')
df['Treatment_Base'] = df['Treatment'].astype(str).str.split('_').str[0]

# Define Symbol and special Color logic
def assign_plot_logic(row):
    treatment = str(row['Treatment']).lower()
    is_control = any(ctrl in treatment for ctrl in ["no_sgrna", "nosgrna"])
    
    if is_control:
        # Controls get a unique label so we can color them black
        color_group = "Control (nosgrna)"
        symbol = "circle" if row['Plate'] == "PLATE6_T1" else "x"
    else:
        # Mutants use their antibiotic name for color
        color_group = row['Treatment_Base']
        symbol = "circle" if row['Plate'] == "PLATE6_T1" else "x"
        
    return pd.Series([color_group, symbol])

df[['Color_Group', 'Symbol_Type']] = df.apply(assign_plot_logic, axis=1)

# Create a color map to force Controls to be Black
unique_treatments = df['Color_Group'].unique()
color_map = {t: px.colors.qualitative.Alphabet[i % 26] for i, t in enumerate(unique_treatments)}
color_map["Control (nosgrna)"] = "#000000"  # Hex for pure black

# 3. DEFINE CHANNELS
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# 4. RUN UMAP
for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Processing UMAP for: {config['name']}...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=10, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df_plot = df.copy()
    df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    fig = px.scatter(
        df_plot, x='UMAP1', y='UMAP2', 
        color='Color_Group',
        symbol='Symbol_Type',
        color_discrete_map=color_map, # Forces control to black
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'Cell_Count': True},
        title=f"UMAP: {config['name']} (Black Controls: Dot=P6, X=P7)",
        template='plotly_white'
    )
    
    # Final styling
    fig.update_traces(marker=dict(size=8, opacity=0.7))
    # Make the Black Controls slightly larger and fully opaque to stand out
    fig.update_traces(marker=dict(size=10, opacity=1.0), selector=dict(marker_color='#000000'))
    
    fig.update_layout(width=1000, height=800, legend_title_text='Treatments & Controls')
    
    save_path = os.path.join(OUTPUT_DIR, f"UMAP_BlackControls2_{config['name'].replace(' ', '_')}.html")
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    fig.show()

In [ ]:
# antibiotica classe anotatie

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
file_path = os.path.join(PROJECT_ROOT, "31march_filtered", "antibiotics_aggregated_wells_mean_juist.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "2aprilAntibiotics", "UMAP_MOA_Groups")
os.makedirs(OUTPUT_DIR, exist_ok=True)

df.columns = [str(c) for c in df.columns]

# --- NEW: Define MOA Mapping ---
moa_map = {
    'vancomycin': 'Cell Wall', 
    'cefalexin': 'Cell Wall', 
    'cefotaxime': 'Cell Wall',
    'rifampicin': 'RNA',
    'ciprofloxacin': 'DNA',
    'gentamycin': 'Ribosome', 
    'kanamycin': 'Ribosome', 
    'tetracycline': 'Ribosome',
    'Control (nosgrna)': 'Control'
}

# 2. PRE-PROCESSING
df['Treatment_Base'] = df['Treatment'].astype(str).str.split('_').str[0]

def assign_plot_logic(row):
    treatment = str(row['Treatment']).lower()
    is_control = any(ctrl in treatment for ctrl in ["no_sgrna", "nosgrna"])
    
    if is_control:
        color_group = "Control (nosgrna)"
    else:
        color_group = row['Treatment_Base']
    
    # Assign the MOA based on the treatment base
    moa = moa_map.get(color_group, 'Unknown')
    symbol = "circle" if row['Plate'] == "PLATE6_T1" else "x"
    return pd.Series([color_group, symbol, moa])

df[['Color_Group', 'Symbol_Type', 'MOA']] = df.apply(assign_plot_logic, axis=1)

# Create a color map for the MOA groups
unique_moas = df['MOA'].unique()
# Using a clear qualitative palette
color_map = {m: px.colors.qualitative.Bold[i % 10] for i, m in enumerate(unique_moas)}
color_map["Control"] = "#000000"  # Force Control MOA to Black

# 3. DEFINE CHANNELS
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# 4. RUN UMAP
for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Processing UMAP for: {config['name']}...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    
    # Keeping your original settings
    embedding = umap.UMAP(n_neighbors=10, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df_plot = df.copy()
    df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    fig = px.scatter(
        df_plot, x='UMAP1', y='UMAP2', 
        color='MOA',  # Color by the biological target
        symbol='Symbol_Type',
        color_discrete_map=color_map,
        hover_name='Treatment', # Allows you to see specific antibiotic on hover
        hover_data={'Plate': True, 'Well_ID': True, 'Color_Group': True},
        title=f"UMAP: {config['name']} grouped by MOA (Black=Control)",
        template='plotly_white'
    )
    
    # Final styling
    fig.update_traces(marker=dict(size=8, opacity=0.7))
    # Make the Black Controls stand out
    fig.update_traces(marker=dict(size=10, opacity=1.0), selector=dict(marker_color='#000000'))
    
    fig.update_layout(width=1000, height=800, legend_title_text='Mechanism of Action')
    
    save_path = os.path.join(OUTPUT_DIR, f"UMAP_MOA_{config['name'].replace(' ', '_')}.html")
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    fig.show()